# 04 — Load OR sample data → gold Delta

| Field | Value |
| ----- | ----- |
| **Sprint** | Sprint 09 — T5.5 |
| **Layer** | `gold.or_schedule` + `gold.or_case` (managed Delta tables) |
| **Source** | `Files/or-samples/*.json` (lakehouse-uploaded T5.4 fixtures; falls back to `data/synthetic/or-samples/*.json` for local CI dry-run) |
| **Contracts** | [DC-OR-SCHEDULE-v1](../../../data/synthetic/schema/dc-or-schedule-v1.schema.json), [DC-OR-CASE-v1](../../../data/synthetic/schema/dc-or-case-v1.schema.json) |
| **Governance** | [ADR-0013 dual-mode residency](../../../docs/adr/0013-temporary-us-region-demo-scope.md), [ADR-0016 gate 2 — no PHI in demo](../../../docs/adr/0016-no-phi-in-mvp-demo-scope.md) |
| **Design spec** | [Sprint 09 v2 §6.2 OR page](../../../docs/superpowers/specs/2026-07-02-sprint-09-v2-refinement-design.md) |

## Purpose

Extract `records[]` from the envelope-format sample fixtures, add gold-layer
governance columns (`_ingested_at`, `_source_system`, `_pseudonymisation_flag`,
`_data_quality`, `_residency_tag`), derive `hospitalId` from the ID prefix
(the source contract carries no top-level `hospitalId`), and write to Delta.

**Sample data only — live OR ingestion is Sprint 10.** These rows exist so
the semantic model (T4.1) and dashboard (T5.1) have concrete facts during
Sprint 09 development.

## Runtime

This notebook is authored for a **Fabric notebook context** where `spark` is
pre-bound. The final write cell is wrapped in a `try/except NameError` so a
local dry-run (no Spark) exits cleanly for CI validation.

In [ ]:
# --- Config -------------------------------------------------------------
target_lakehouse = 'lh_ihzhhpf_sit'
or_samples_path = 'Files/or-samples'                       # lakehouse-relative source folder
source_system = 'or-samples-fixture'

# Local-mode source paths (repo-relative) - used only as a CI dry-run fallback
# when no Spark session is bound (see cell below, guarded by NameError).
from pathlib import Path
REPO = Path.cwd().parents[2] if (Path.cwd().name == 'reference') else Path.cwd()
SCHEDULE_PATH = REPO / 'data' / 'synthetic' / 'or-samples' / 'or_schedule.json'
CASE_PATH = REPO / 'data' / 'synthetic' / 'or-samples' / 'or_case.json'
print(f'or_samples_path (lakehouse): {or_samples_path}')
print(f'schedule (local fallback): {SCHEDULE_PATH}')
print(f'case (local fallback):     {CASE_PATH}')

In [ ]:
# --- Load envelopes -----------------------------------------------------
# Fabric context: read the envelope JSON files from the lakehouse Files/ mount.
# Local dry-run: NameError on `spark` falls back to the repo-relative fixtures
# so CI can execute this cell without a Spark session and still exit 0.
import json

try:
    schedule_text = '\n'.join(
        r.value for r in spark.read.option('multiline', 'true').text(f'{or_samples_path}/or_schedule.json').collect()  # type: ignore[name-defined]
    )
    case_text = '\n'.join(
        r.value for r in spark.read.option('multiline', 'true').text(f'{or_samples_path}/or_case.json').collect()  # type: ignore[name-defined]
    )
    schedule_envelope = json.loads(schedule_text)
    case_envelope = json.loads(case_text)
except NameError:
    schedule_envelope = json.loads(SCHEDULE_PATH.read_text(encoding='utf-8'))
    case_envelope = json.loads(CASE_PATH.read_text(encoding='utf-8'))

assert schedule_envelope['contractId'] == 'DC-OR-SCHEDULE-v1'
assert case_envelope['contractId'] == 'DC-OR-CASE-v1'

schedule_records = schedule_envelope['records']
case_records = case_envelope['records']
print(f'Loaded {len(schedule_records)} slot records, {len(case_records)} case-event records')
print(f'Unique cases: {len({r["caseId"] for r in case_records})}')

In [ ]:
# --- Add governance columns --------------------------------------------
# hospitalId is derived from the ID prefix because DC-OR-SCHEDULE-v1 /
# DC-OR-CASE-v1 do not carry it at record level (residency is per-region).
from datetime import datetime, timezone

INGESTED_AT = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
RESIDENCY_TAG = 'US-West'  # ADR-0013 demo scope; PROD flips to CH-North via override

HOSPITAL_PREFIX_MAP = {'USZ': 'H_USZ', 'LUKS': 'H_LUKS', 'SZB': 'H_SZB'}

def _derive_hospital(id_value: str) -> str:
    # ORS-USZ-000001 or ORT-USZ-01 or ORC-USZ-000001
    parts = id_value.split('-')
    if len(parts) < 3:
        return 'H_UNKNOWN'
    return HOSPITAL_PREFIX_MAP.get(parts[1], 'H_UNKNOWN')

def _enrich(records: list, hospital_source_field: str) -> list:
    out = []
    for r in records:
        enriched = dict(r)
        enriched['hospitalId'] = _derive_hospital(r[hospital_source_field])
        enriched['_ingested_at'] = INGESTED_AT
        enriched['_source_system'] = source_system
        enriched['_pseudonymisation_flag'] = True
        enriched['_data_quality'] = 'explicit'
        enriched['_residency_tag'] = RESIDENCY_TAG
        out.append(enriched)
    return out

schedule_gold = _enrich(schedule_records, 'orSlotId')
case_gold = _enrich(case_records, 'caseId')

print('Hospital distribution (slots):')
from collections import Counter
for k, v in sorted(Counter(r['hospitalId'] for r in schedule_gold).items()):
    print(f'  {k}: {v}')
print('Governance columns present:',
      sorted(k for k in schedule_gold[0].keys() if k.startswith('_') or k == 'hospitalId'))

In [ ]:
# --- Write to gold Delta (Fabric-context only) --------------------------
# Local dry-run: NameError on `spark` is caught so CI can execute this cell
# without a Spark session and still exit 0.
try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS gold")  # type: ignore[name-defined]
    # Build an EXPLICIT schema. Some contract columns (e.g.
    # or_case.turnoverPredecessorCaseId) are null in every record; letting
    # spark.createDataFrame infer would raise CANNOT_DETERMINE_TYPE on the
    # un-inferable NullType. Default all-null columns to nullable StringType.
    from pyspark.sql.types import (StructType, StructField, StringType,
                                   LongType, BooleanType, ArrayType)

    def _spark_type(value):
        # bool must precede int: bool is a subclass of int in Python.
        if isinstance(value, bool):
            return BooleanType()
        if isinstance(value, int):
            return LongType()
        if isinstance(value, list):
            return ArrayType(StringType())
        return StringType()

    def _build_schema(records):
        keys = []
        for r in records:
            for k in r:
                if k not in keys:
                    keys.append(k)
        return StructType([
            StructField(k, _spark_type(next((r[k] for r in records if r.get(k) is not None), None)), True)
            for k in keys
        ])

    df_schedule = spark.createDataFrame(schedule_gold, schema=_build_schema(schedule_gold))  # type: ignore[name-defined]
    df_case = spark.createDataFrame(case_gold, schema=_build_schema(case_gold))  # type: ignore[name-defined]
    (df_schedule.write
        .format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable('gold.or_schedule'))
    (df_case.write
        .format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable('gold.or_case'))
    print(f'Wrote {df_schedule.count()} slots -> gold.or_schedule')
    print(f'Wrote {df_case.count()} case events -> gold.or_case')
except NameError:
    print('LOCAL DRY-RUN: `spark` not bound. Skipping Delta write.')
    print(f'  would write {len(schedule_gold)} slots -> gold.or_schedule')
    print(f'  would write {len(case_gold)} case events -> gold.or_case')